##Задача регрессии для SI (Selectivity Index)
Аналогично предыдущим задачам, мы будем предсказывать логарифмированный индекс селективности ( pSI=log10(SI) ) и сравнивать эффективность моделей.

In [8]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 26.8 MB/s eta 0:00:00


In [9]:
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import KFold, cross_validate
import optuna

In [10]:


# Load the dataset
file_path = '/content/cleaned_molecular_data.csv'
df = pd.read_csv(file_path)

# Display basic information
print("Dataset Shape:", df.shape)
display(df.head())
print(df.info())

Dataset Shape: (966, 214)


,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,is_outlier
0,6.239374,175.482382,28.125000,5.094096,5.094096,0.387225,0.387225,0.417362,42.928571,384.652,...,0,0,0,0,0,0,0,3,0,1
1,0.771831,5.402819,7.000000,3.961417,3.961417,0.533868,0.533868,0.462473,45.214286,388.684,...,0,0,0,0,0,0,0,3,0,1
2,223.808778,161.142320,0.720000,2.627117,2.627117,0.543231,0.543231,0.260923,42.187500,446.808,...,0,0,0,0,0,0,0,3,0,1
3,1.705624,107.855654,63.235294,5.097360,5.097360,0.390603,0.390603,0.377846,41.862069,398.679,...,0,0,0,0,0,0,0,4,0,1
4,107.131532,139.270991,1.300000,5.150510,5.150510,0.270476,0.270476,0.429038,36.514286,466.713,...,0,0,0,0,0,0,0,0,0,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Columns: 214 entries, IC50, mM to is_outlier
dtypes: float64(107), int64(107)
memory usage: 1.6 MB
None


In [11]:
# 1. Подготовка данных для SI
target_si = 'SI'
exclude_cols_si = ['IC50, mM', 'CC50, mM', 'SI', 'IC50_above_median', 'CC50_above_median', 'SI_above_median', 'SI_above_8', 'is_outlier']
# SI расчитывается из CC50 IC50 , я понял задачу как построение регресии без этих предикторов (так как это целевые переменные задачи 1 и 2)
# , если же их использование возможно , то из списка выше их нужно было бы исключить
# Признаки
relevant_features_si = [col for col in df.select_dtypes(include=[np.number]).columns if col not in exclude_cols_si]

# Убираем NaN в SI (если есть) и логарифмируем
df_si = df.dropna(subset=[target_si]).copy()
df_si = df_si[df_si[target_si] > 0] # SI должен быть больше 0 для логарифмирования

X_si = df_si[relevant_features_si]
y_si_log = np.log10(df_si[target_si])

X_train_si, X_test_si, y_train_si, y_test_si = train_test_split(X_si, y_si_log, test_size=0.2, random_state=42)

print(f"Samples for SI task: {len(df_si)}")

Samples for SI task: 966


### Базовые модели (стандартные параметры) для SI

In [14]:
standard_models = {
    "GBR (Default)": GradientBoostingRegressor(random_state=42),
    "RF (Default)": RandomForestRegressor(random_state=42),
    "LGBM (Default)": lgb.LGBMRegressor(random_state=42, verbosity=-1)
}
baseline_si = []
for name, model in standard_models.items():
    model.fit(X_train_si, y_train_si)
    tr_p= model.predict(X_train_si)
    te_p = model.predict(X_test_si)
    baseline_si.append({
        "Model": name,
        "Train RMSE": np.sqrt(mean_squared_error(y_train_si, tr_p)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test_si, te_p)),
        "Test R2": r2_score(y_test_si, te_p)
    })
display(pd.DataFrame(baseline_si))

,Model,Train RMSE,Test RMSE,Test R2
0,GBR (Default),0.483795,0.666920,0.196010
1,RF (Default),0.400265,0.664826,0.201051
2,LGBM (Default),0.369449,0.691354,0.136020


In [ ]:
def run_optuna_si(model_class, n_trials=200):
    def objective(trial):
        if model_class == GradientBoostingRegressor:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
                'max_depth': trial.suggest_int('max_depth', 3, 7),
                'random_state': 42
            }
        elif model_class == RandomForestRegressor:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 5, 15),
                'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
                'random_state': 42
            }
        else: # LGBM
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 20, 60),
                'random_state': 42, 'verbosity': -1
            }

        model = model_class(**params)
        kf = KFold(n_splits=4, shuffle=True, random_state=42)
        cv = cross_validate(model, X_train_si, y_train_si, cv=kf, scoring='neg_mean_squared_error', return_train_score=True)
        val_rmse = np.mean(np.sqrt(-cv['test_score']))
        train_rmse = np.mean(np.sqrt(-cv['train_score']))
        return val_rmse - 2 * abs(val_rmse - train_rmse)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

print("Tuning models for SI...")
best_gbr_si = run_optuna_si(GradientBoostingRegressor)
best_rf_si = run_optuna_si(RandomForestRegressor)
best_lgbm_si = run_optuna_si(lgb.LGBMRegressor)

[I 2026-05-23 10:35:09,252] A new study created in memory with name: no-name-679ccf35-c10f-442a-91cb-d230e1ad5c49


Tuning models for SI...


[I 2026-05-23 10:35:25,909] Trial 0 finished with value: 0.5829875372366307 and parameters: {'n_estimators': 147, 'learning_rate': 0.005006860441970198, 'max_depth': 4}. Best is trial 0 with value: 0.5829875372366307.
[I 2026-05-23 10:35:37,963] Trial 1 finished with value: 0.5088255365917665 and parameters: {'n_estimators': 72, 'learning_rate': 0.009001334857397954, 'max_depth': 6}. Best is trial 0 with value: 0.5829875372366307.
[I 2026-05-23 10:36:09,311] Trial 2 finished with value: 0.06771929998858095 and parameters: {'n_estimators': 270, 'learning_rate': 0.06258524858828403, 'max_depth': 4}. Best is trial 0 with value: 0.5829875372366307.
[I 2026-05-23 10:36:20,009] Trial 3 finished with value: 0.4288813408681923 and parameters: {'n_estimators': 120, 'learning_rate': 0.034355490277287265, 'max_depth': 3}. Best is trial 0 with value: 0.5829875372366307.
[I 2026-05-23 10:37:06,344] Trial 4 finished with value: -0.0022875138547648888 and parameters: {'n_estimators': 266, 'learning_r

In [ ]:
best_gbr_si

{'n_estimators': 50, 'learning_rate': 0.00500800000856754, 'max_depth': 3}

In [ ]:
best_lgbm_si

{'n_estimators': 50, 'learning_rate': 0.005000320565186036, 'num_leaves': 45}

In [ ]:
best_rf_si

{'n_estimators': 57, 'max_depth': 5, 'max_features': 'log2'}

In [17]:
best_rf_si={'n_estimators': 57, 'max_depth': 5, 'max_features': 'log2'}
best_lgbm_si={'n_estimators': 50, 'learning_rate': 0.005000320565186036, 'num_leaves': 45}
best_gbr_si={'n_estimators': 50, 'learning_rate': 0.00500800000856754, 'max_depth': 3}

In [18]:
# 3. Сравнение и финальная оценка SI
si_results = []
models_si = {
    "GBR": GradientBoostingRegressor(**best_gbr_si, random_state=42),
    "RF": RandomForestRegressor(**best_rf_si, random_state=42),
    "LGBM": lgb.LGBMRegressor(**best_lgbm_si, random_state=42, verbosity=-1)
}

for name, model in models_si.items():
    model.fit(X_train_si, y_train_si)
    p_tr = model.predict(X_train_si)
    p_te = model.predict(X_test_si)
    si_results.append({
        "Model": name,
        "Train RMSE": np.sqrt(mean_squared_error(y_train_si, p_tr)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test_si, p_te)),
        "Test R2": r2_score(y_test_si, p_te)
    })

display(pd.DataFrame(si_results))



,Model,Train RMSE,Test RMSE,Test R2
0,GBR,0.717108,0.723575,0.053612
1,RF,0.591767,0.662566,0.206474
2,LGBM,0.690771,0.710958,0.086328


видно что нам удалоьс незначительно улучшить метрику модели на тестовых данных , но зато теперь предсказания моделей стабильны на трейне и тесте, нет переобучения.

In [19]:
# Оценка на отфильтрованных данных
df_si_f = df_si[df_si['is_outlier'] == 1].copy()
X_si_f = df_si_f[relevant_features_si]
y_si_f = np.log10(df_si_f[target_si])
X_tr_f, X_te_f, y_tr_f, y_te_f = train_test_split(X_si_f, y_si_f, test_size=0.2, random_state=42)

final_model_si = RandomForestRegressor(**best_rf_si, random_state=42)
final_model_si.fit(X_tr_f, y_tr_f)
f_preds_si = final_model_si.predict(X_te_f)

print(f"\nFinal SI Results (Outliers Removed):")
print(f"RMSE: {np.sqrt(mean_squared_error(y_te_f, f_preds_si)):.4f}")
print(f"R2:   {r2_score(y_te_f, f_preds_si):.4f}")


Final SI Results (Outliers Removed):
RMSE: 0.6855
R2:   0.2370


в данной задачен исключение выбросов не дало улчшения скора, возможно выбросы имеют смысл для данной задачи